In [ ]:
!pip install timm

In [ ]:
import os
import torch
import torch.nn as nn
import timm
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from PIL import Image
from torchvision import transforms
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# --- 1. Configuration ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Use your standardized 80/10/10 split path
BASE_DIR = '/content/CNN_ViT_Hybrid_Vit_Research_Pneumonia_4_Classification/Train_Test_Split_Vit_Data'
train_data_dir = os.path.join(BASE_DIR, 'train')
val_data_dir = os.path.join(BASE_DIR, 'val')

# --- 2. Initialize ConViT ---
# 'convit_tiny' is highly efficient and matches your other 'tiny/small' baselines
model_convit = timm.create_model('convit_tiny', pretrained=True, num_classes=4)
model_convit.to(device)

# --- 3. Preprocessing ---
# ConViT expects standard ImageNet normalization
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# --- 4. Dataset Class ---
class PneumoniaDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.classes = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
        self.class_to_idx = {name: i for i, name in enumerate(self.classes)}

        for class_name in self.classes:
            path = os.path.join(data_dir, class_name)
            for img in os.listdir(path):
                if img.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.image_paths.append(os.path.join(path, img))
                    self.labels.append(self.class_to_idx[class_name])

    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(self.labels[idx])

# Create Dataloaders
train_dataloader = DataLoader(PneumoniaDataset(train_data_dir, data_transforms), batch_size=32, shuffle=True)
val_dataloader = DataLoader(PneumoniaDataset(val_data_dir, data_transforms), batch_size=32, shuffle=False)

# --- 5. Optimizer & Loss ---
optimizer = AdamW(model_convit.parameters(), lr=5e-5, weight_decay=0.05)
criterion = nn.CrossEntropyLoss()

In [ ]:
epochs = 25
print(f"Starting ConViT Training on {device}...")

for epoch in range(epochs):
    model_convit.train()
    train_loss, correct, total = 0, 0, 0
    for inputs, labels in train_dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model_convit(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    # Validation Phase
    model_convit.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for v_inputs, v_labels in val_dataloader:
            v_inputs, v_labels = v_inputs.to(device), v_labels.to(device)
            v_outputs = model_convit(v_inputs)
            _, v_pred = v_outputs.max(1)
            val_total += v_labels.size(0)
            val_correct += v_pred.eq(v_labels).sum().item()

    print(f"ConViT Epoch {epoch+1}/{epochs} | Loss: {train_loss/len(train_dataloader):.4f} | "
          f"Train Acc: {100.*correct/total:.2f}% | Val Acc: {100.*val_correct/val_total:.2f}%")

In [ ]:
model_convit.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in val_dataloader:
        outputs = model_convit(inputs.to(device))
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

# Confusion Matrix
plt.figure(figsize=(10, 8))
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Spectral', xticklabels=train_dataloader.dataset.classes, yticklabels=train_dataloader.dataset.classes)
plt.title('ConViT Confusion Matrix')
plt.show()

print("\nConViT Classification Report:\n")
print(classification_report(all_labels, all_preds, target_names=train_dataloader.dataset.classes))